<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Find the Best Correction

**Choose one clock correction by solving a derivative equation, then verify its minimum.**

We keep four fixed clock observations and the squared-error objective. A derivative equation gives a candidate correction; completing the square checks it against every real correction.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Reuse the clock decision

The observed errors are $-2,-1,-1,0$ minutes. A negative error means the clock is slow.
Choose one correction $q$ in minutes and add it to every reading. For observation $i=1,\ldots,4$, let $d_i$ be the recorded error and $y_i$ the remaining error:

$$
y_i=d_i+q.
$$

For example, $q=1$ changes the errors to $-1,0,0,1$ min. We compare corrections using the sum of squared remaining errors. Call this score $\phi(q)$:

$$
\phi(q)=(q-2)^2+(q-1)^2+(q-1)^2+q^2.
$$

The score has units of min². Our task is to make it as small as possible.
Every real correction is allowed, so the problem is **unconstrained**. A hardware limit on $q$ would change that.

The definitions below collect the fixed observations, error and score calculations, derivative calculation, and graph with its slider.


In [ ]:
import numpy as np
import sys
import matplotlib

# Fixed observations, measured in minutes
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])


def simulate_clock(correction, observed_errors=OBSERVED_ERRORS):
    return np.asarray(observed_errors, dtype=float) + float(correction)


def objective_function(remaining_errors):
    return float(np.sum(np.asarray(remaining_errors) ** 2))


def evaluate_clock(correction, observed_errors=OBSERVED_ERRORS):
    response = simulate_clock(correction, observed_errors)
    return {
        "decision": float(correction),
        "response": response,
        "objective": objective_function(response),
        "derivative": float(2.0 * np.sum(response)),
    }


def solve_clock_correction(observed_errors=OBSERVED_ERRORS):
    return float(-np.mean(observed_errors))


def _pyplot(*, interactive=False):
    # Match the course's Jupyter/VS Code and browser backend pattern.
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry
            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
GRAY = "#a6a6a6"


def show_clock_analysis(correction=0.0, *, interactive=False, display=True):
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(1, 2, figsize=(12, 6.4 if interactive else 5.1))
    figure.suptitle("One correction minimizes the total squared clock error", fontsize=14)
    if interactive:
        figure.subplots_adjust(left=0.08, right=0.98, top=0.78,
                               bottom=0.28, wspace=0.29)
    status = figure.text(0.5, 0.88, "", ha="center", fontsize=10)
    positions = np.arange(1, 5)
    axes[0].bar(positions - 0.18, OBSERVED_ERRORS, width=0.36,
                color=BLUE, alpha=0.45, label="Observed error")
    bars = axes[0].bar(positions + 0.18, OBSERVED_ERRORS, width=0.36,
                       color=ORANGE, label="Current corrected error")
    axes[0].axhline(0.0, color=GRAY, linewidth=1)
    axes[0].set(xlabel="Comparison number", ylabel="Clock error (min)",
                xticks=positions, ylim=(-3.5, 3.5), title="The same q shifts every error")
    axes[0].legend(loc="upper left", fontsize=8)

    curve_corrections = np.linspace(-1.0, 3.0, 301)
    curve_scores = [evaluate_clock(value)["objective"] for value in curve_corrections]
    axes[1].plot(curve_corrections, curve_scores, color=BLUE, label="Score")
    optimum = evaluate_clock(solve_clock_correction())
    axes[1].scatter([optimum["decision"]], [optimum["objective"]],
                    color=TEAL, marker="D", s=90, label="Analytical minimum", zorder=4)
    marker = axes[1].scatter([], [], color=ORANGE, s=70,
                             label="Current decision", zorder=5)
    tangent, = axes[1].plot([], [], color=ORANGE, linestyle="--", label="Local slope")
    axes[1].set(xlabel="Correction q (min)", ylabel="Squared-error score (min²)",
                xlim=(-1.1, 3.1), ylim=(0, 22), title="The slope is zero at q = 1")
    axes[1].legend(loc="upper center", fontsize=8)
    for axis in axes:
        axis.grid(alpha=0.25)

    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider
        slider_axis = figure.add_axes([0.29, 0.13, 0.58, 0.035])
        sliders["correction"] = Slider(slider_axis, "Correction q (min)", -1.0, 3.0,
                                       valinit=correction, valstep=0.05, color=BLUE)
    else:
        figure.tight_layout(rect=(0, 0.08, 1, 0.83))

    figure.text(0.5, 0.035, "Shown interval and slider step control the display; q may be any real number.",
                ha="center", fontsize=9)
    figure._analysis_state = {}

    def refresh(_=None):
        value = sliders["correction"].val if sliders else correction
        record = evaluate_clock(value)
        for bar, error in zip(bars, record["response"]):
            bar.set_height(error)
        marker.set_offsets([[value, record["objective"]]])
        nearby = np.array([value - 0.25, value + 0.25])
        tangent.set_data(nearby, record["objective"] + record["derivative"] * (nearby - value))
        status.set_text(f"q = {value:.2f} min | score = {record['objective']:.2f} min² | "
                        f"slope = {record['derivative']:.2f} | minimum score = 2.00 min²")
        figure._analysis_state.update(current=record, optimum=optimum)
        figure.canvas.draw_idle()

    figure._analysis_sliders = sliders
    figure._analysis_refresh = refresh
    for slider in sliders.values():
        slider.on_changed(refresh)
    refresh()
    if display:
        plt.show()
    return figure


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Use the slope to find a candidate

Expand the four squares:

$$
\phi(q)=4q^2-8q+6.
$$

Differentiate each term. The derivative of $4q^2$ is $8q$, the derivative of $-8q$ is $-8$, and the constant contributes zero:

$$
\phi'(q)=8q-8.
$$

The derivative tells us how the score changes when we increase $q$ a little.

| Correction | Derivative | Effect of a small increase in $q$ |
|:---|:---|:---|
| $q=0$ | $-8$ | Score decreases |
| $q=1$ | $0$ | Check whether this is a minimum |
| $q=2$ | $8$ | Score increases |

A **stationary point** has derivative zero. At a differentiable unconstrained minimum, the derivative must be zero; otherwise a small move could lower the score. Solve:

$$
8q-8=0\quad\Longrightarrow\quad q=1.
$$

This gives a candidate. We still need to check that it is a minimum.
For this squared-error problem, the correction cancels the mean error: $(-2-1-1+0)/4=-1$ min, so add 1 min.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Calculate the answer by cancelling the mean observed error. The result should agree with the derivative equation.


In [ ]:
mean_error = float(np.mean(OBSERVED_ERRORS))
best_correction = -mean_error
best_result = evaluate_clock(best_correction)
print("Mean observed error:", mean_error, "min")
print("Best correction:", best_correction, "min")
print("Derivative there:", best_result["derivative"])
print("Score there:", best_result["objective"], "min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Check that the answer is best

A **local minimum** is best among nearby decisions. A **global minimum** is best among all allowed decisions. Ties are allowed in both definitions.

The second derivative describes how the slope changes. Here $\phi''(q)=8>0$: the curve bends upward, and its slope changes from negative to positive at $q=1$.

We can also check every real correction at once by rewriting the score:

$$
\phi(q)=4(q-1)^2+2.
$$

A square cannot be negative. The score is therefore at least 2 min², and reaches 2 only when $q=1$.
This proves a **unique global minimum**. We write the best correction as $q^\star=1$; the star marks the optimal decision.
The remaining errors are $-1,0,0,1$ min, so even the best single correction does not remove every error.

The left panel compares the fixed observations with the corrected errors. The right panel shows the same decision on the score curve; the short dashed line has slope $\phi'(q)$. Compare the current orange marker with the teal diamond at the analytically verified minimum.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-1_unconstrained_optimization_analytical_methods/assets/clock_correction_analytic.png" alt="Clock errors before and after a one-minute correction, alongside the score curve with its unique analytical minimum at q equals one." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The curve is sampled for drawing; the minimum comes from the algebra, not a grid search.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The blue slider controls the correction shown on the error bars and score curve. Its finite interval and step are display settings; the optimization domain remains $\mathbb R$.


In [ ]:
clock_explorer = show_clock_analysis(correction=0.0, interactive=True)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · A flat slope can also be a maximum

**A zero derivative is not enough.** For comparison, reverse the clock score and call it $\psi(q)=-\phi(q)$:

$$
\psi(q)=-4(q-1)^2-2.
$$

Its derivative is also zero at $q=1$. But the minus sign makes this the top of the curve. For example, $\psi(0)=-6$, $\psi(1)=-2$, and $\psi(2)=-6$. Minimizing this reversed score would reward larger errors.

Compare the nearby values on both sides of each orange point. Both dashed tangents are horizontal, but only the left point is lower than its neighbors. These are sketches of the two shapes near $q=1$; their heights are not a shared numerical scale.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-1_unconstrained_optimization_analytical_methods/assets/flat_slope_two_shapes.png" alt="An upward-opening curve has a horizontal tangent at a minimum, while a downward-opening curve has a horizontal tangent at a maximum. Zero slope alone does not distinguish them." width="400" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

For a smooth function of one variable, the second derivative at a stationary point gives a useful check:

| Second derivative | Conclusion |
|:---|:---|
| Positive | Local minimum |
| Negative | Local maximum |
| Zero | This test does not decide |

For example, $\rho(q)=(q-1)^4$ has both first and second derivatives zero at 1. It still has a minimum there because a fourth power cannot be negative.

The solution minimizes squared error on the four fixed observations. It does not establish error on future readings.
